# Data Unification and Integration

Orden de prioridad
1. Pegar Info del acervo(lo que se puede rescatar)
1.1 Hay que meterle al scraper del acervo los ids de los pdfs que no se pudieron extraer. (1380) y los meta al detalle.resultados
2. API Confidence: 1
3. API Confidence: 0.9
4. Pdfs

bd=x manual y cerrar

#### Carga

In [90]:
import pandas as pd

df = pd.read_csv("6.marcia_mexico_pre_data_merge.csv")

df = df[['id', 'title', 'applicationNumber', 'registrationNumber',
       'applicationDate', 'publicationDt', 'registrationDate', 'expiryDate',
       'appType', 'classes', 'Addr_x', 'Cry', 'Name', 'email', 'pdf_links',
       'Addr_id', 'municipality', 'state']]

# preprocesamiento
#dtypes

df['applicationNumber'] = pd.to_numeric(df['applicationNumber'], errors='coerce').astype('Int64')
df['registrationNumber'] = pd.to_numeric(df['registrationNumber'], errors='coerce').astype('Int64')

#df = df.columns.to_lowercase()


print(df.columns)

print(df.info())

Index(['id', 'title', 'applicationNumber', 'registrationNumber',
       'applicationDate', 'publicationDt', 'registrationDate', 'expiryDate',
       'appType', 'classes', 'Addr_x', 'Cry', 'Name', 'email', 'pdf_links',
       'Addr_id', 'municipality', 'state'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103094 entries, 0 to 103093
Data columns (total 18 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id                  103094 non-null  object
 1   title               99177 non-null   object
 2   applicationNumber   103090 non-null  Int64 
 3   registrationNumber  103090 non-null  Int64 
 4   applicationDate     103090 non-null  object
 5   publicationDt       103021 non-null  object
 6   registrationDate    103090 non-null  object
 7   expiryDate          103090 non-null  object
 8   appType             103090 non-null  object
 9   classes             103090 non-null  object
 10  Addr_x      

/tmp/ipykernel_5971/764189514.py:3: DtypeWarning: Columns (9,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("6.marcia_mexico_pre_data_merge.csv")


### Merging prep

#### 1.acervo

In [91]:
# #revisar si acervo tiene filas en las cuales registrationNumber o applicationNumber en combiancion esten repetidas

# print(acervo.duplicated(subset=['registrationNumber', 'applicationNumber']).sum())

# #ver subset de duplicados
# sub_acervo = acervo[acervo.duplicated(subset=['registrationNumber', 'applicationNumber'], keep=False)]

In [92]:
import pandas as pd
import json

file_path = '3.detalle_resultados.jsonl'
data = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
             # ← Aquí usamos las DOS columnas
        try:
            record = json.loads(line)
            
            # Accedemos correctamente a las secciones
            dg = record.get("detalle", {}).get("datos_generales", {})
            ap = record.get("detalle", {}).get("apoderado", {})
            
            row = {
                "registrationNumber": record.get("registrationNumber"),
                
                # ← CORREGIDO AQUÍ
                "applicationNumber": dg.get("datos_generales_Número_de_expediente"),
                
                "titular_Población": dg.get("titular_Población"),
                "titular_E-mail": dg.get("titular_E-mail"),
                "establecimiento_Población": dg.get("establecimiento_Población"),
                
                "apoderado_Población": ap.get("apoderado_Población"),
                "apoderado_E-mail": ap.get("apoderado_E-mail")
            }
            
            data.append(row)
            
        except json.JSONDecodeError:
            print("Error al decodificar una línea")
            continue

# Crear DataFrame
acervo = pd.DataFrame(data)

acervo['registrationNumber'] = pd.to_numeric(acervo['registrationNumber'], errors='coerce').astype('Int64')
acervo['applicationNumber']   = pd.to_numeric(acervo['applicationNumber'],   errors='coerce').astype('Int64')


#eliminar duplicados
acervo = acervo.drop_duplicates(subset=['registrationNumber', 'applicationNumber'], keep='first')

merged = pd.merge(
    df,
    acervo[['registrationNumber', 
            'applicationNumber',
            'titular_Población', 
            'titular_E-mail', 
            'apoderado_E-mail']],      
    on=['registrationNumber', 'applicationNumber'],  
    how='left'
)

merged['bd'] = 'Nan'                    # valor por defecto
merged.loc[merged['titular_Población'].notna(), 'bd'] = 'acervo'

split_cols = merged['titular_Población'].str.split(',', n=1, expand=True)

# Asignar a las columnas existentes
merged['municipality'] = split_cols[0].str.strip()          # izquierda de la coma
merged['state']       = split_cols[1].str.strip() if split_cols.shape[1] > 1 else None

merged = merged.drop(columns=['titular_Población'])    
merged['bd'] = merged['bd'].replace(['Nan', 'nan', 'None', '', ' ', '-'], np.nan)

merged.loc[merged['Addr_id'] == "x", 'bd'] = 'x'

merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103094 entries, 0 to 103093
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id                  103094 non-null  object
 1   title               99177 non-null   object
 2   applicationNumber   103090 non-null  Int64 
 3   registrationNumber  103090 non-null  Int64 
 4   applicationDate     103090 non-null  object
 5   publicationDt       103021 non-null  object
 6   registrationDate    103090 non-null  object
 7   expiryDate          103090 non-null  object
 8   appType             103090 non-null  object
 9   classes             103090 non-null  object
 10  Addr_x              103090 non-null  object
 11  Cry                 103090 non-null  object
 12  Name                103082 non-null  object
 13  email               9834 non-null    object
 14  pdf_links           103077 non-null  object
 15  Addr_id             103089 non-null  object
 16  mu

In [93]:
cols = ['municipality', 'state', 'titular_E-mail', 'apoderado_E-mail', 'bd']

merged[cols] = (
    merged.sort_values('Addr_id')  # importante para consistencia
      .groupby('Addr_id')[cols]
      .transform('ffill')
      .groupby(merged['Addr_id'])[cols]
      .transform('bfill')
)


In [94]:
# print(merged['bd'].value_counts())

# print("*********************************************")
# print(merged['state'].value_counts())

# print("*********************************************")
# print(merged.shape)

In [95]:
print(merged['bd'].value_counts(dropna=False ))

bd
NaN       97016
acervo     6074
x             4
Name: count, dtype: int64


#### 2. API Confidence: 1

creo que este deberia ser el ultimo por que aunque tiene confianza 1, pdfs seria de mejor confianza igual que 0.9 por que los estoy haciendo manualmente

In [96]:
api_1 = pd.read_csv("6.data_integretation_checkpoint_data/confidence_score_1_0.csv")

#mergear con main
mask = merged['bd'].isna() 

temp = pd.merge(
    merged[mask][['id']],                    # solo tomamos el id de las filas Nan
    api_1[['id', 'municipality', 'state']],  # traemos solo lo necesario
    on='id',
    how='left'
)
merged.loc[mask, 'municipality'] = temp['municipality'].values
merged.loc[mask, 'state']        = temp['state'].values

merged.loc[
    (merged['bd'].isna()) & 
    (merged['state'].notna()), 
    'bd'
] = 'api_1'


In [97]:
cols = ['municipality', 'state', 'titular_E-mail', 'apoderado_E-mail', 'bd']

merged[cols] = (
    merged.sort_values('Addr_id')  # importante para consistencia
      .groupby('Addr_id')[cols]
      .transform('ffill')
      .groupby(merged['Addr_id'])[cols]
      .transform('bfill')
)


In [98]:
print(merged['bd'].value_counts(dropna=False ))

bd
api_1     63901
NaN       33115
acervo     6074
x             4
Name: count, dtype: int64


### 4. PDF 100% Confidence

In [99]:
pdf1 = pd.read_csv("8.data_pdf_scrapped.csv")

mask = merged['bd'].isna()

temp = pd.merge(
    merged[mask][['id']],
    pdf1[['id', 'municipality', 'state', 'titular_E-mail',
           'municipio_notificaciones', 'entidad_notificaciones', 'bd']],
    on='id',
    how='left'
).set_index(merged[mask].index)   # ← importante: alinear índices

# Actualizar
columns_to_update = ['municipality', 'state', 'titular_E-mail', 
                     'municipio_notificaciones', 'entidad_notificaciones', 'bd']

merged.loc[mask, columns_to_update] = temp[columns_to_update]

In [100]:
cols = ['municipality', 'state', 'titular_E-mail', 'apoderado_E-mail', 'bd']

merged[cols] = (
    merged.sort_values('Addr_id')  # importante para consistencia
      .groupby('Addr_id')[cols]
      .transform('ffill')
      .groupby(merged['Addr_id'])[cols]
      .transform('bfill')
)


In [101]:
print(merged['bd'].value_counts(dropna=False ))

bd
api_1     63901
pdf       22723
NaN       10364
acervo     6074
x            32
Name: count, dtype: int64


#### 3. API = .90

In [102]:
api_9 = pd.read_csv(
    "6.data_integretation_checkpoint_data/confidence_score_0_9.csv",
    encoding='latin1'          # or 'iso-8859-1'
)
#mergear con main
mask = merged['bd'].isna() 

temp = pd.merge(
    merged[mask][['id']],                    # solo tomamos el id de las filas Nan
    api_9[['id', 'municipality', 'state']],  # traemos solo lo necesario
    on='id',
    how='left'
)
merged.loc[mask, 'municipality'] = temp['municipality'].values
merged.loc[mask, 'state']        = temp['state'].values

merged.loc[
    (merged['bd'].isna()) & 
    (merged['state'].notna()), 
    'bd'
] = 'api_9'


In [103]:
cols = ['municipality', 'state', 'titular_E-mail', 'apoderado_E-mail', 'bd']

merged[cols] = (
    merged.sort_values('Addr_id')  # importante para consistencia
      .groupby('Addr_id')[cols]
      .transform('ffill')
      .groupby(merged['Addr_id'])[cols]
      .transform('bfill')
)

In [ ]:
print(final_df['bd'].value_counts(dropna=False))

bd
api_1     63901
pdf       22723
api_9     10341
acervo     6074
x            55
Name: count, dtype: int64


In [2]:
### Limpieza y transformación antes de guardar documento final

final_df = merged.copy()


#bd: if NaN == "X"
final_df["bd"] = final_df["bd"].fillna("x")

# fill na email
final_df['email'] = final_df['email'].fillna(final_df['titular_E-mail'])

#drop unnecessary columns
final_df = final_df.drop(columns=['titular_E-mail','Addr_id', 'Cry'])




NameError: name 'merged' is not defined

In [3]:
NORM_NOTIF = {
    "CIUDAD DE MÉXICO":                  "CIUDAD DE MEXICO",
    "NUEVO LEÓN":                         "NUEVO LEON",
    "QUERÉTARO":                          "QUERETARO",
    "ESTADO DE MÉXICO":                   "ESTADO DE MEXICO",
    "YUCATÁN":                            "YUCATAN",
    "CDMX":                               "CIUDAD DE MEXICO",
    "MICHOACÁN":                          "MICHOACAN",
    "MICHOACÁN DE OCAMPO":                "MICHOACAN",
    "MICHOACAN DE OCAMPO":                "MICHOACAN",
    "DISTRITO FEDERAL":                   "CIUDAD DE MEXICO",
    "DF":                                 "CIUDAD DE MEXICO",
    "CDMX CIUDAD DE MEXICO":              "CIUDAD DE MEXICO",
    "CIUDAD DE MEXICO CDMX":              "CIUDAD DE MEXICO",
    "CIUDAD DE MÉXCO":                    "CIUDAD DE MEXICO",
    "COAHUILA DE ZARAGOZA":               "COAHUILA",
    "VERACRUZ DE IGNACIO DE LA LLAVE":    "VERACRUZ",
    "SAN LUIS POTOSÍ":                    "SAN LUIS POTOSI",
    "MÉXICO":                             "ESTADO DE MEXICO",
    "MEXICO":                             "ESTADO DE MEXICO",
    "GTO":                                "GUANAJUATO",
    "SIN":                                "SINALOA",
    "NAY":                                "NAYARIT",
    "TAMPS":                              "TAMAULIPAS",
    "LOS MOCHIS":                         "SINALOA",
    # el valor con 100+ ceros al final
    **{k: "COAHUILA" for k in final_df["entidad_notificaciones"].dropna().unique()
       if isinstance(k, str) and k.startswith("COAHUILA 0")},
}

final_df["entidad_notificaciones"] = (
    final_df["entidad_notificaciones"]
    .apply(lambda v: NORM_NOTIF.get(v.strip().upper(), v.strip().upper())
           if isinstance(v, str) else v)
)

print(final_df["entidad_notificaciones"].value_counts())

ESTADOS_VALIDOS = {
    "AGUASCALIENTES", "BAJA CALIFORNIA", "BAJA CALIFORNIA SUR", "CAMPECHE",
    "CHIAPAS", "CHIHUAHUA", "CIUDAD DE MEXICO", "COAHUILA", "COLIMA",
    "DURANGO", "ESTADO DE MEXICO", "GUANAJUATO", "GUERRERO", "HIDALGO",
    "JALISCO", "MICHOACAN", "MORELOS", "NAYARIT", "NUEVO LEON", "OAXACA",
    "PUEBLA", "QUERETARO", "QUINTANA ROO", "SAN LUIS POTOSI", "SINALOA",
    "SONORA", "TABASCO", "TAMAULIPAS", "TLAXCALA", "VERACRUZ", "YUCATAN",
    "ZACATECAS",
}

print(f"\nFuera del catálogo: {(~final_df['entidad_notificaciones'].isin(ESTADOS_VALIDOS) & final_df['entidad_notificaciones'].notna()).sum()}")
print(final_df[~final_df['entidad_notificaciones'].isin(ESTADOS_VALIDOS) & final_df['entidad_notificaciones'].notna()]['entidad_notificaciones'].value_counts())

NameError: name 'final_df' is not defined

In [4]:
NORM_STATE = {
    # CIUDAD DE MEXICO
    "Ciudad de México":                         "CIUDAD DE MEXICO",
    "CIUDAD DE MÉXICO":                         "CIUDAD DE MEXICO",
    "CDMX":                                     "CIUDAD DE MEXICO",
    "D.F.":                                     "CIUDAD DE MEXICO",
    "DISTRITO FEDERAL":                         "CIUDAD DE MEXICO",
    "CIUDAD DE  MEXICO":                        "CIUDAD DE MEXICO",
    # ESTADO DE MEXICO
    "Estado de México":                         "ESTADO DE MEXICO",
    "ESTADO DE MÉXICO":                         "ESTADO DE MEXICO",
    "EDO. DE MEX.":                             "ESTADO DE MEXICO",
    "MÉXICO":                                   "ESTADO DE MEXICO",
    "MEXICO":                                   "ESTADO DE MEXICO",
    "EDOMEX (ZONA METROPOLITANA)":              "ESTADO DE MEXICO",
    "ESTADO DE MEXICO53370":                    "ESTADO DE MEXICO",
    "BENITO JUAREZ, EDO. DE MEX.":             "ESTADO DE MEXICO",
    "NAUCALPAN, EDO. DE MEX.":                 "ESTADO DE MEXICO",
    "ATLACOMULCO, EDO. DE MEX.":               "ESTADO DE MEXICO",
    "TOLUCA, EDO. DE MEX.":                    "ESTADO DE MEXICO",
    "TECAMAC DE FELIPE VILLANUEVA, ESTADO DE MEXICO": "ESTADO DE MEXICO",
    "ZAPOPAN":                                  "JALISCO",   # ciudad → estado
    # JALISCO
    "Jalisco":                                  "JALISCO",
    "JAL.":                                     "JALISCO",
    "JALISCO.":                                 "JALISCO",
    ": JALISCO":                                "JALISCO",
    "USMAJAC, JAL.":                            "JALISCO",
    "TLAJOMULCO DE ZUÑ, JALISCO":              "JALISCO",
    # NUEVO LEON
    "Nuevo León":                               "NUEVO LEON",
    "NUEVO LEÓN":                               "NUEVO LEON",
    "NUEVO LEÓN.":                              "NUEVO LEON",
    "N.L.":                                     "NUEVO LEON",
    "MONTERREY, NUEVO LEÓN":                    "NUEVO LEON",
    "MONTERREY, N.L.":                          "NUEVO LEON",
    # GUANAJUATO
    "Guanajuato":                               "GUANAJUATO",
    "GTO.":                                     "GUANAJUATO",
    "URIANGATO, GTO.":                          "GUANAJUATO",
    # BAJA CALIFORNIA
    "Baja California":                          "BAJA CALIFORNIA",
    "B.C.":                                     "BAJA CALIFORNIA",
    "BAJA CALIFORNIA NORTE":                    "BAJA CALIFORNIA",
    "SAN JOSE DEL CABO, B.C.":                 "BAJA CALIFORNIA",
    # BAJA CALIFORNIA SUR
    "Baja California Sur":                      "BAJA CALIFORNIA SUR",
    "B.C.S.":                                   "BAJA CALIFORNIA SUR",
    "BAJA CALIFRONIA SUR":                      "BAJA CALIFORNIA SUR",
    "B.C.S., B.C.S.":                          "BAJA CALIFORNIA SUR",
    "LOS CABOS, B.C.S.":                       "BAJA CALIFORNIA SUR",
    # PUEBLA
    "Puebla":                                   "PUEBLA",
    "PUE.":                                     "PUEBLA",
    "TEPEYAHUALCO, PUEBLA":                     "PUEBLA",
    "TEPEYAHUALCO, PUE.":                       "PUEBLA",
    "CUAUTLANCINGO, PUEBLA.":                   "PUEBLA",
    "CUAUTLANCINGO, PUE.":                      "PUEBLA",
    # QUERETARO
    "Querétaro":                                "QUERETARO",
    "QUERÉTARO":                                "QUERETARO",
    "QRO.":                                     "QUERETARO",
    "QUERETARO, QRO.":                          "QUERETARO",
    # SINALOA
    "Sinaloa":                                  "SINALOA",
    "SIN.":                                     "SINALOA",
    "AHOME, SIN.":                              "SINALOA",
    "CULIACAN, SIN.":                           "SINALOA",
    "ANGOSTURA, SIN.":                          "SINALOA",
    # QUINTANA ROO
    "Quintana Roo":                             "QUINTANA ROO",
    "Q. ROO":                                   "QUINTANA ROO",
    "Q, ROO":                                   "QUINTANA ROO",
    "Q,ROO.":                                   "QUINTANA ROO",
    "Q.ROO":                                    "QUINTANA ROO",
    "CANCÚN, QUINTANA ROO":                     "QUINTANA ROO",
    "CANCUN, QUINTANA ROO":                     "QUINTANA ROO",
    "CANCUN., QUINTANA ROO.":                   "QUINTANA ROO",
    "BENITO JUAREZ, Q. ROO":                    "QUINTANA ROO",
    "SOLIDARIDAD, Q. ROO":                      "QUINTANA ROO",
    "SOLIDARIDAD, QUINTANA ROO":                "QUINTANA ROO",
    "QUINTANA ROOMEXUCI":                       "QUINTANA ROO",
    # YUCATAN
    "Yucatán":                                  "YUCATAN",
    "YUCATÁN":                                  "YUCATAN",
    "YUC.":                                     "YUCATAN",
    "YUCATAN (YUC)":                            "YUCATAN",
    # CHIHUAHUA
    "Chihuahua":                                "CHIHUAHUA",
    "CHIH.":                                    "CHIHUAHUA",
    "CHIHAUAHUA":                               "CHIHUAHUA",
    # MICHOACAN
    "Michoacán":                                "MICHOACAN",
    "MICHOACÁN":                                "MICHOACAN",
    "MICH.":                                    "MICHOACAN",
    "MICH. DE OCAMPO":                          "MICHOACAN",
    "MICHOACÁN DE OCAMPO":                      "MICHOACAN",
    "MORELIA":                                  "MICHOACAN",
    # SONORA
    "Sonora":                                   "SONORA",
    "SON.":                                     "SONORA",
    # AGUASCALIENTES
    "Aguascalientes":                           "AGUASCALIENTES",
    "AGS.":                                     "AGUASCALIENTES",
    "JESUS MARIA , AGUASCALIENTES":             "AGUASCALIENTES",
    # VERACRUZ
    "Veracruz":                                 "VERACRUZ",
    "VER.":                                     "VERACRUZ",
    "VER, VERACRUZ":                            "VERACRUZ",
    "VERACRÚZ":                                 "VERACRUZ",
    "VERACRUZ DE IGNACIO DE LA LLAVE":          "VERACRUZ",
    "VER. DE IGNACIO DE LA LLAVE":              "VERACRUZ",
    "ATOYAC, VERACRUZ":                         "VERACRUZ",
    "LAZARO CARDENAS, VER.":                    "VERACRUZ",
    # COAHUILA
    "Coahuila":                                 "COAHUILA",
    "COAHUILA DE ZARAGOZA":                     "COAHUILA",
    "COAH.":                                    "COAHUILA",
    "COAH. DE ZARAGOZA":                        "COAHUILA",
    "COAHUILA DE ZARAGOZA":                     "COAHUILA",
    # TAMAULIPAS
    "Tamaulipas":                               "TAMAULIPAS",
    "TAMPS.":                                   "TAMAULIPAS",
    # MORELOS
    "Morelos":                                  "MORELOS",
    "MOR.":                                     "MORELOS",
    # HIDALGO
    "Hidalgo":                                  "HIDALGO",
    "HGO.":                                     "HIDALGO",
    "YOLOTEPEC, HGO.":                          "HIDALGO",
    # SAN LUIS POTOSI
    "San Luis Potosí":                          "SAN LUIS POTOSI",
    "SAN LUIS POTOSÍ":                          "SAN LUIS POTOSI",
    "S.L.P.":                                   "SAN LUIS POTOSI",
    # OAXACA
    "Oaxaca":                                   "OAXACA",
    "OAX.":                                     "OAXACA",
    "OAXACA DE JUÁREZ, OAXACA":                 "OAXACA",
    "SAN BALTAZAR GUELAVILA, OAX.":             "OAXACA",
    # TABASCO
    "Tabasco":                                  "TABASCO",
    "TAB.":                                     "TABASCO",
    "CENTRO, TABASCO":                          "TABASCO",
    "CENTRO, TAB.":                             "TABASCO",
    "VILLAHERMOSA":                             "TABASCO",
    # CHIAPAS
    "Chiapas":                                  "CHIAPAS",
    "CHIS.":                                    "CHIAPAS",
    # NAYARIT
    "Nayarit":                                  "NAYARIT",
    "NAY.":                                     "NAYARIT",
    # COLIMA
    "Colima":                                   "COLIMA",
    "COL.":                                     "COLIMA",
    # DURANGO
    "Durango":                                  "DURANGO",
    "DGO.":                                     "DURANGO",
    "DUR.":                                     "DURANGO",
    # ZACATECAS
    "Zacatecas":                                "ZACATECAS",
    "ZAC.":                                     "ZACATECAS",
    # CAMPECHE
    "Campeche":                                 "CAMPECHE",
    "CAMP.":                                    "CAMPECHE",
    # GUERRERO
    "Guerrero":                                 "GUERRERO",
    "GRO.":                                     "GUERRERO",
    "ESTADO DE GUERRERO":                       "GUERRERO",
    # TLAXCALA
    "Tlaxcala":                                 "TLAXCALA",
    "TLAX.":                                    "TLAXCALA",
}

NO_MEXICO_STATE = {
    "FLORIDA", "FL.", "INDIANA", "NEW JERSEY", "CALIFORNIA", "TEXAS",
    "PENSILVANIA", "NUEVA YORK", "NY.", "DELAWARE", "SPAIN", "MADRID",
    "CHILE", "PANAMA", "BOGOTÁ", "BENOS AIRES", "HUBEI", "GUANGDONG",
    "XINTIAN COUNTY, HUANAN PROVINCE", "SP", "MN", "JC", "NUEVO",
    "SOLTERO A", "CASADO A", "PLEASE SELECT", "vieja", "CIUDAD",
    "SAN PEDRO HUAQUILPAN",
}

def normalizar_state(v):
    if not isinstance(v, str):
        return v
    return NORM_STATE.get(v.strip(), v.strip().upper())

final_df["state"] = final_df["state"].apply(normalizar_state)

mask = final_df["state"].apply(
    lambda v: isinstance(v, str) and v.strip().upper() not in ESTADOS_VALIDOS
)
final_df.loc[mask, "bd"] = "x"

mask_coah = final_df["state"].str.contains("COAHUILA", case=False, na=False) & (final_df["bd"] == "x")
final_df.loc[mask_coah, "bd"] = "api_1"

NameError: name 'final_df' is not defined

In [5]:
RENAME_COLS = {
    "id":                       "id",
    "title":                    "marca",
    "applicationNumber":        "num_expediente",
    "registrationNumber":       "num_registro",
    "applicationDate":          "fecha_solicitud",
    "publicationDt":            "fecha_publicacion",
    "registrationDate":         "fecha_registro",
    "expiryDate":               "fecha_vencimiento",
    "appType":                  "tipo_solicitud",
    "classes":                  "clases",
    "Addr_x":                   "domicilio",
    "Name":                     "titular",
    "email":                    "email_titular",
    "pdf_links":                "url_pdf",
    "municipality":             "municipio_titular",
    "state":                    "estado_titular",
    "apoderado_E-mail":         "email_apoderado",
    "bd":                       "bd_extranjero",
    "municipio_notificaciones": "municipio_notificaciones",
    "entidad_notificaciones":   "estado_notificaciones",
}

final_df = final_df.rename(columns=RENAME_COLS)
print(final_df.columns.tolist())

NameError: name 'final_df' is not defined

In [173]:
# Muestra TODO completo y permite scrollear
with pd.option_context('display.max_rows', None):
    print(final_df['bd_extranjero'].value_counts(dropna=False))

bd_extranjero
api_1     64130
pdf       22723
api_9     10111
acervo     6060
x            70
Name: count, dtype: int64


In [174]:
# primera letra mayuscula
columnas = ["municipio_titular", "domicilio"]

for col in columnas:
    if col in final_df.columns:
        final_df[col] = final_df[col].astype(str).str.capitalize()

In [176]:
sub_final_df = final_df[final_df['estado_titular'] == "BAJA CALIFORNIA"] 

In [ ]:
tramites = pd.read_csv("tramites_2023.csv")

In [178]:
COLS_ORDER = [
    "num_expediente",
    "num_registro",
    "tipo_solicitud",
    "marca",
    "clases",
    "titular",
    "email_titular",
    "municipio_titular",
    "estado_titular",
    "domicilio",
    "email_apoderado",
    "municipio_notificaciones",
    "estado_notificaciones",
    "fecha_solicitud",
    "fecha_publicacion",
    "fecha_registro",
    "fecha_vencimiento",
    "id",
    "url_pdf",
    "bd_extranjero",
]

final_df = final_df[COLS_ORDER]

sub_final_df = sub_final_df[COLS_ORDER]

In [179]:
import csv

final_df.to_csv(
    "registros_2023_mex.csv",
    index=False,                    # No guardar el índice
    encoding='utf-8-sig'
)

import csv

sub_final_df.to_csv(
    "registros_2023_bc.csv",
    index=False,                    # No guardar el índice
    encoding='utf-8-sig'
)